# 대구 골든타임 정책분석 EDA

## tl;dr

- 2026.07.18 정책 릴리스는 150개 행정동, 25개 기관, 9개 후보와 5,100개 도로 경로를 포함하며 누락 경로는 0개입니다.
- VDI는 일반 차량 도로 ETA와 취약인구를 결합한 상대 우선순위 신호입니다.
- 후보 최적화 결과는 현장 조사 전에 검토 범위를 좁히는 비교 자료이며 시설 신설 확정안이 아닙니다.

## Context & Methods

이 노트북은 단일 `policy_release.json`에 묶인 2026.07.18 검증본을 탐색적으로 점검합니다. 분석 결과는 의료적 위험 확률, 실제 수용 가능성 또는 시설 신설 확정안을 의미하지 않습니다.

In [ ]:
import hashlib
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import seaborn as sns

WORKING_DIR = Path.cwd().resolve()
release_candidates = [
    WORKING_DIR / 'datasets/policy_release.json',
    WORKING_DIR.parent / 'datasets/policy_release.json',
    WORKING_DIR / 'frontend/public/data/policy_release.json',
]
RELEASE_PATH = next((path for path in release_candidates if path.exists()), None)
if RELEASE_PATH is None:
    raise FileNotFoundError('python/fetch_policy_release.py를 먼저 실행해 datasets/policy_release.json을 준비하세요.')
release = json.loads(RELEASE_PATH.read_text(encoding='utf-8'))
metadata = release['metadata']

def payload_sha256(value):
    encoded = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()

expected_routes = metadata['district_count'] * (metadata['resource_count'] + metadata['candidate_count'])
assert metadata['district_count'] == len(release['vulnerability']['features']) == 150
assert metadata['resource_count'] == len(release['hospitals']) == 25
assert metadata['candidate_count'] == len(release['candidates']) == 9
assert metadata['route_count'] == metadata['successful_route_count'] == expected_routes == 5100
assert metadata['missing_route_count'] == 0
for name in ('hospitals', 'vulnerability', 'candidates', 'candidate_trace', 'optimization'):
    assert payload_sha256(release[name]) == metadata['content_sha256'][name], f'{name} SHA-256 불일치'
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
font_candidates = ('Malgun Gothic', 'AppleGothic', 'Noto Sans CJK KR', 'Noto Sans KR', 'NanumGothic')
selected_font = next((font for font in font_candidates if font in available_fonts), 'DejaVu Sans')
plt.rc('font', family=selected_font)
plt.rcParams['axes.unicode_minus'] = False
metadata

## Data: 행정동 분석 테이블 구성

현재 공개 VDI는 일반 차량 도로 ETA와 취약인구를 결합합니다. 행정동 중심점은 실제 환자 위치나 구급차 출발점이 아닙니다.

In [ ]:
rows = []
for feature in release['vulnerability']['features']:
    props = feature['properties']
    rows.append({
        'adm_nm': props['adm_nm'],
        'senior_population': props['65세이상_인구'],
        'pediatric_population': props['0~9세_인구'],
        'vulnerable_population': props['취약인구'],
        'straight_distance_km': props['min_dist_to_hospital'],
        'road_eta_minutes': props['travel_time_minutes'],
        'vdi': props['vulnerability_index'],
        'vdi_norm': props['vdi_norm'],
        'nearest_tier': props['nearest_hospital_tier'],
    })
df = pd.DataFrame(rows)
df.describe().T

## Results: VDI 분포와 상대 경계

빨간 점선은 의료적 절대 임계값이 아니라 현재 분석본 내부 상위 25% 상대 경계입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['vdi'], bins=20, kde=True, ax=axes[0], color='salmon')
axes[0].axvline(metadata['risk_threshold'], color='darkred', linestyle='--', label='상위 25% 상대 경계')
axes[0].set(title='도로 ETA 기반 VDI 분포', xlabel='VDI')
axes[0].legend()
sns.histplot(df['road_eta_minutes'], bins=20, kde=True, ax=axes[1], color='skyblue')
axes[1].set(title='일반 차량 도로 ETA 분포', xlabel='분')
plt.tight_layout()

## Results: 변수 관계와 상위 지역

상관계수는 현재 150개 행정동에서 관찰된 기술통계이며 인과관계를 증명하지 않습니다.

In [ ]:
corr_columns = ['senior_population', 'pediatric_population', 'vulnerable_population', 'straight_distance_km', 'road_eta_minutes', 'vdi']
sns.heatmap(df[corr_columns].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('현재 분석본의 변수 상관관계')
plt.tight_layout()
df.nlargest(10, 'vdi')[['adm_nm', 'vdi', 'road_eta_minutes', 'vulnerable_population']]

## Results: 후보 조합 결과

p-median·MCLP 결과는 분류된 후보군 내부의 수학적 비교입니다. 대구 전역의 전역 최적해, 실제 시설 건립 효과 또는 환자 수용 성과가 아닙니다.

In [ ]:
summary_rows = []
for mode, results in release['optimization']['results'].items():
    for result in results:
        mclp = result['mclp_30min_optimum']
        p_median = result['p_median_optimum']
        summary_rows.append({
            'mode': mode,
            'candidate_count': result['facility_count'],
            'p_median_weighted_eta': p_median['weighted_average_eta_minutes'],
            'mclp_30min_coverage_ratio': mclp['covered_30min_ratio'],
            'mclp_30min_covered_population': mclp['covered_30min_population'],
        })
pd.DataFrame(summary_rows)

## Takeaways

- 릴리스 규모와 구성요소 내용 해시가 모두 검증돼 같은 분석본을 읽고 있음을 확인했습니다.
- VDI와 상관관계는 현재 150개 행정동의 기술통계로만 해석하며 인과관계나 의료적 절대 위험을 뜻하지 않습니다.
- p-median·MCLP 결과는 정해진 후보군 내부 비교이므로 실제 정책 결정에는 최신 원천 확인과 현장 검토가 추가로 필요합니다.